## 🧱 Özel Ortam

---

Bu görevde, derste örnek olarak kullanılan **ızgara tabanlı navigasyon ortamının** birebir aynısını tasarlayıp uygulayacaksınız.

- Ajan ızgaranın bir köşesinde başlar.
- Amaç karşı köşeye ulaşmaktır.
- Izgarada ajanın kaçınması gereken engeller veya "delikler" bulunabilir.

---

### 🎯 Amaçlar

- 📐 **Ortamı Tanımlayın**: ajanın başlangıçtan hedefe doğru hareket ettiği bir ızgara dünyası oluşturun.
- ⚙️ **Ortam Dinamiklerini Uygulayın**: hareket ve ödül ataması kurallarını programlayın (örn. deliklere düşmek için ceza, hedefe ulaşmak için ödül).
- 👁️ **Gözlem ve Eylemleri Ayarlayın**:
    - **Gözlem alanı** → Ajanın algıladığı şeyler (örn. ızgaradaki pozisyonu)  
    - **Eylem alanı** → Ajanın yapabileceği şeyler (örn. yukarı, aşağı, sola, sağa hareket)
- 🖼️ **Render Metodu Ekleyin**: ortamı görselleştirmek için `.render()` fonksiyonu ekleyin — hata ayıklama ve ajanın davranışını anlama için faydalıdır.
- 🧩 **Özel Özellikler Ekleyin**: derste ele alınan engeller, çukurlar veya diğer özellikleri dahil ederek ortamınızı daha dinamik ve gerçekçi hale getirin.

---
Bu görev için ihtiyacımız olan tüm paketleri içe aktararak başlayalım:

In [1]:
import time
import numpy as np
import matplotlib.pyplot as plt
from typing import Optional, Dict, Tuple


import gymnasium as gym
from gymnasium import spaces
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3 import DQN
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.env_util import make_vec_env

---

### 🧩 Bölüm 1: Özel Ortam Sınıfı Oluşturma

Bu bölümde, kısmen yazılmış bir sınıfı tamamlayarak özel ortamınızı uygulayacaksınız.  
Yapı, Gymnasium'un ortam oluşturma için standart formatını takip eder.

### 🛠️ Yapacaklarınız

- Bazı temel metotları önceden yazılmış bir şablon sağlanmıştır.  
- `# CODE HERE` yazdığını gördüğünüz her yerde, eksik mantığı doldurmanız gerekir.  
- Her adımda size rehberlik etmesi için satır içi ipuçları verilmiştir.

### 🎯 Hedefiniz

- Sınıfı gerçek bir Gymnasium ortamı gibi davranacak şekilde tamamlayın.  
- Şunları yapmalı:
  - Geçerli bir gözlem ve eylem alanı tanımlamalı  
  - Ajan hareketini ve geçişleri ele almalı  
  - Ödülleri döndürmeli ve `done` bayraklarını uygun şekilde güncellemeli  
  - Çalışan bir `.reset()` ve `.step()` metodu içermeli  
  - İsteğe bağlı olarak görselleştirme için basit bir `.render()` içermeli

🧠 Her metodu anlamak için zaman ayırın — özellikle durum geçişlerinin ve ödüllerin nasıl yönetildiğini. RL ortamları burada canlanır.

📚 Ortam yapısı ve en iyi uygulamalar hakkında ayrıntılı adımlar için resmi [Gymnasium özel ortam kılavuzuna](https://gymnasium.farama.org/introduction/create_custom_env/) başvurun.

In [19]:
class CustomGridEnv(gym.Env):

    def __init__(self):
        super().__init__()

        self.size = 5

        self.observation_space = spaces.Dict({
            "agent": spaces.Box(low=0, high=self.size - 1, shape=(2,), dtype=np.int32),
            "target": spaces.Box(low=0, high=self.size - 1, shape=(2,), dtype=np.int32),
        })

        self.action_space = spaces.Discrete(4)

        self.agent_position = np.array([0, 0], dtype=np.int32)
        self.goal_position = np.array([self.size - 1, self.size - 1], dtype=np.int32)

        self.holes = [
            np.array([1, 1], dtype=np.int32),
            np.array([2, 3], dtype=np.int32),
            np.array([3, 1], dtype=np.int32)
        ]

    def _get_obs(self):
        return {
            "agent": self.agent_position.copy(),
            "target": self.goal_position.copy()
        }

    def _get_info(self):
        return {
            "distance": np.sum(np.abs(self.agent_position - self.goal_position))
        }

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        self.agent_position = np.array([0, 0], dtype=np.int32)

        observation = self._get_obs()
        info = self._get_info()

        return observation, info

    def step(self, action):

        reward = -1
        done = False

        if action == 0:
            self.agent_position[0] -= 1
        elif action == 1:
            self.agent_position[0] += 1
        elif action == 2:
            self.agent_position[1] -= 1
        elif action == 3:
            self.agent_position[1] += 1

        self.agent_position = np.clip(
            self.agent_position,
            0,
            self.size - 1
        )

        for hole in self.holes:
            if np.array_equal(self.agent_position, hole):
                reward = -10
                done = True

        if np.array_equal(self.agent_position, self.goal_position):
            reward = 10
            done = True

        observation = self._get_obs()
        info = self._get_info()

        return observation, reward, done, False, info

    def render(self, mode="human"):

        grid = np.full((self.size, self.size), ".", dtype=str)

        for hole in self.holes:
            grid[hole[0], hole[1]] = "H"

        grid[self.goal_position[0], self.goal_position[1]] = "G"
        grid[self.agent_position[0], self.agent_position[1]] = "A"

        print(grid)

👇 Önceki bölümü doğru tamamladığınızı test etmek için aşağıdaki hücreyi çalıştırın. Doğru yaptıysanız, ajanın bir bölümün sonu olan hedefe veya deliğe ulaşana kadar ortamınızda rastgele hareket ettiğini göreceksiniz.

In [20]:
# Create an instance of the custom environment
env = CustomGridEnv()

# Reset the environment to its initial state and get the initial observation and info
obs, info = env.reset()
print("Initial Observation:", obs)  # Display the initial position of the agent and the target
print("Initial Info:", info)  # Display additional information such as the distance from the target

# Loop through a maximum of 100 steps
for _ in range(100):
    action = env.action_space.sample()  # Randomly sample an action from the action space
    obs, reward, done, info, _ = env.step(action)  # Apply the action and get the results
    env.render()  # Render the current state of the environment to visualize the agent's position

    # Print the current state, reward received, whether the episode is done, and any additional info
    print(f"State: {obs}, Reward: {reward}, Done: {done}, Info: {info}")

    # If the episode is finished (agent reached the goal or fell into a hole), exit the loop
    if done:
        break


Initial Observation: {'agent': array([0, 0], dtype=int32), 'target': array([4, 4], dtype=int32)}
Initial Info: {'distance': np.int64(8)}
[['.' '.' '.' '.' '.']
 ['A' 'H' '.' '.' '.']
 ['.' '.' '.' 'H' '.']
 ['.' 'H' '.' '.' '.']
 ['.' '.' '.' '.' 'G']]
State: {'agent': array([1, 0], dtype=int32), 'target': array([4, 4], dtype=int32)}, Reward: -1, Done: False, Info: False
[['.' '.' '.' '.' '.']
 ['.' 'H' '.' '.' '.']
 ['A' '.' '.' 'H' '.']
 ['.' 'H' '.' '.' '.']
 ['.' '.' '.' '.' 'G']]
State: {'agent': array([2, 0], dtype=int32), 'target': array([4, 4], dtype=int32)}, Reward: -1, Done: False, Info: False
[['.' '.' '.' '.' '.']
 ['.' 'H' '.' '.' '.']
 ['.' '.' '.' 'H' '.']
 ['A' 'H' '.' '.' '.']
 ['.' '.' '.' '.' 'G']]
State: {'agent': array([3, 0], dtype=int32), 'target': array([4, 4], dtype=int32)}, Reward: -1, Done: False, Info: False
[['.' '.' '.' '.' '.']
 ['.' 'H' '.' '.' '.']
 ['A' '.' '.' 'H' '.']
 ['.' 'H' '.' '.' '.']
 ['.' '.' '.' '.' 'G']]
State: {'agent': array([2, 0], dtype

---
## Bölüm 2: DQN Eğitimi 🤖

Bu bölümde, özel ortamınızı başlatacak ve Stable Baselines3 kütüphanesini kullanarak bir DQN ajanı ile etkileşim için hazırlayacaksınız. Temel görev, ortamınızın kütüphanenin gereksinimleriyle uyumlu olduğundan emin olmaktır, bu da onu uygun şekilde sarmalamayı içerir.

#### 📝 İzlenecek adımlar

1. 🧱 **Özel Ortamı Başlatın**: özel ortam sınıfınızın bir örneğini oluşturun.
2. 🔁 **SB3 Uyumluluğunu Sağlayın**: Stable Baselines3'ten `make_vec_env` fonksiyonunu kullanarak ortamınızı sarın, böylece kütüphanenin vektörleştirilmiş ortam gereksinimleriyle uyumlu hale getirin.
3. ⚙️ **DQN Ajanını Yapılandırın ve Eğitin**: DQN ajanını uygun hiperparametrelerle kurun ve ortamınızda eğitin.
4. 📊 **Eğitim İlerlemesini İzleyin**: Ajanın öğrenme ilerlemesini ve performansını zaman içinde gözlemlemek için günlükleme ve izleme uygulayın.
5. 💾 **Modeli kaydedin**: Eğitim sonrasında modelinizi kaydedin.

In [21]:
from stable_baselines3 import DQN
from stable_baselines3.common.env_util import make_vec_env

# Ortam oluştur
env = CustomGridEnv()

# Stable Baselines3 için vectorized env
vec_env = make_vec_env(lambda: CustomGridEnv(), n_envs=1)

# DQN modeli oluştur
model = DQN(
    "MultiInputPolicy",
    vec_env,
    verbose=1,
    learning_rate=0.001,
    buffer_size=10000,
    learning_starts=100,
    batch_size=32,
    gamma=0.99,
    train_freq=4,
    target_update_interval=100
)

# Eğitimi başlat
model.learn(total_timesteps=10000)

# Modeli kaydet
model.save("custom_grid_dqn")

Using cpu device
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 9.5      |
|    ep_rew_mean      | -18.5    |
|    exploration_rate | 0.964    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 2933     |
|    time_elapsed     | 0        |
|    total_timesteps  | 38       |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 7.75     |
|    ep_rew_mean      | -16.8    |
|    exploration_rate | 0.941    |
| time/               |          |
|    episodes         | 8        |
|    fps              | 2697     |
|    time_elapsed     | 0        |
|    total_timesteps  | 62       |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 8.75     |
|    ep_rew_mean      | -17.8    |
|    exploration_rate | 0.9      |
| time/               |          |
|  

---
## 🎮 Bölüm 3: Eğitilmiş modelinizi kullanın

Eğitilmiş DQN modelinizi yükleyecek ve ortamda karar vermek için kullanacaksınız, başlangıç noktasından hedefe delikleri kaçınarak nasıl gittiğini gözlemleyeceksiniz. Bu sadece eğitimin etkinliğini doğrulamanıza izin vermeyecek, aynı zamanda ajanın karar verme sürecini görsel olarak yorumlamanıza da olanak sağlayacaktır.

### İzlenecek adımlar 📝
1. 💾 **Eğitilmiş Modeli Yükleyin**: Eğitim aşamasında kaydedilen DQN modelini alın.
2. 🔄 **Ortamı Sıfırlayın**: Navigasyon görevini sıfırdan başlatmak için ortamı başlatın.
3. 🧠 **Navigasyon Simülasyonunu Çalıştırın**: Modeli kullanarak ortamın durumlarına dayalı eylemleri tahmin edin ve ajanın attığı her adımı görselleştirin.
4. 👀 **Her Adımı Görselleştirin**: Ajanın pozisyonunu, hedefi ve herhangi bir engeli veya deliği gösteren ızgaranın basit bir görselleştirmesini uygulayın.
5. 📝 **Ajan Davranışını Analiz Edin**: Ajanın hedefe ulaşma yeteneğini gözlemleyin ve not edin ve delikleri ne kadar etkili bir şekilde kaçındığını görün.

Kodun bir kısmı zaten orada ve `# CODE HERE` yorumunu gördüğünüz her yerde kod doldurmanız gerekiyor. Başlamanız için bazı ipuçları bulacaksınız.

In [23]:
# Load the model
from stable_baselines3 import DQN
import numpy as np

# modeli yükle
model = DQN.load("custom_grid_dqn")

# ortam oluştur
env = CustomGridEnv()

obs, info = env.reset()

for _ in range(200):

    # model aksiyon tahmini yapar
    action, _states = model.predict(obs)

    # ortamda aksiyonu uygula
    obs, rewards, done, _, info = env.step(action)

    # ızgara oluştur
    grid = np.full((5, 5), ".")

    # ajan koordinatı
    agent_x = obs["agent"][0]
    agent_y = obs["agent"][1]

    # hedef koordinatı
    goal_x = obs["target"][0]
    goal_y = obs["target"][1]

    # delikler
    for hole in env.holes:
        grid[hole[0]][hole[1]] = "H"

    # ajan
    grid[agent_x][agent_y] = "A"

    # hedef
    grid[goal_x][goal_y] = "G"

    print(f"State: {obs}, Reward: {rewards}, Done: {done}, Info: {info}")

    print("----- GRID -----")

    for row in grid:
        print(" ".join(row))

    print()

    if done:
        print("Episode Finished")

        obs, info = env.reset()


obs = env.reset()  # Reset the environment to the initial state and get the first observation

for _ in range(200):  # Loop through a maximum of 200 steps (or until the episode ends)
    # Use the model to predict the next action to be taken, based on the current observation
    # Execute the chosen action in the environment, receive the next state and reward, and check if the episode is done
    pass  # YOUR CODE HERE

    # Initialize a 3x3 grid filled with spaces to represent the environment visually
    grid = np.full((3,3 ), fill_value=' ')

    # Extract coordinates for the agent, goal, and hole using the current observation
    agent_x, agent_y = obs['agent'][0][0],obs['agent'][0][1]
    goal_x, goal_y = obs['target'][0][0],obs['target'][0][1]
    hole_x, hole_y = 2,1  # The hole's position is fixed in the environment

    # Update the grid with the agent's, goal's, and hole's positions
    grid[agent_x][agent_y] = 'A'  # Mark the agent's position with 'A'
    grid[goal_x][goal_y] = 'G'  # Mark the goal position with 'G'
    grid[hole_x][hole_y] = 'H'  # Mark the hole position with 'H'

    # Print the current state, reward received, if the episode is done, and any additional info
    print(f"State: {obs}, Reward: {rewards}, Done: {done}, Info: {info}")

    # Print the grid visually in the console
    print("+---" * 3 + "+")  # Top border of the grid
    for row in grid:
        print("|" + "|".join(f" {cell} " for cell in row) + "|")  # Print each row with cells separated by '|'
        print("+---" * 3 + "+")  # Separator border after each row

    # If the episode is finished (either the goal is reached or the agent fell into the hole), reset the environment
    pass  # YOUR CODE HERE

State: {'agent': array([1, 0], dtype=int32), 'target': array([4, 4], dtype=int32)}, Reward: -1, Done: False, Info: {'distance': np.int64(7)}
----- GRID -----
. . . . .
A H . . .
. . . H .
. H . . .
. . . . G

State: {'agent': array([1, 1], dtype=int32), 'target': array([4, 4], dtype=int32)}, Reward: -10, Done: True, Info: {'distance': np.int64(6)}
----- GRID -----
. . . . .
. A . . .
. . . H .
. H . . .
. . . . G

Episode Finished
State: {'agent': array([1, 0], dtype=int32), 'target': array([4, 4], dtype=int32)}, Reward: -1, Done: False, Info: {'distance': np.int64(7)}
----- GRID -----
. . . . .
A H . . .
. . . H .
. H . . .
. . . . G

State: {'agent': array([1, 1], dtype=int32), 'target': array([4, 4], dtype=int32)}, Reward: -10, Done: True, Info: {'distance': np.int64(6)}
----- GRID -----
. . . . .
. A . . .
. . . H .
. H . . .
. . . . G

Episode Finished
State: {'agent': array([1, 0], dtype=int32), 'target': array([4, 4], dtype=int32)}, Reward: -1, Done: False, Info: {'distance': np

TypeError: tuple indices must be integers or slices, not str

In [24]:
import os
os.getcwd()

'C:\\Users\\RIZA'

In [25]:
import os

for item in os.listdir():
    print(item)

.anaconda
.cache
.conda
.continuum
.copilot
.edevletconf
.git
.gitconfig
.idlerc
.ipynb_checkpoints
.ipython
.jupyter
.keras
.lesshst
.matplotlib
.ms-ad
.pdfbox.cache
.pytest_cache
.python_history
.sertifikadeposu
.spyder-py3
.uki
.virtual_documents
.vscode
.vscode-shared
01-Ham-or-Spam.ipynb
02-Movie-reviews.ipynb
3D Objects
akia.ini
anaconda3
AppData
Application Data
BAYES TEOREMI - WEATHER FORECAST.ipynb
Belgelerim
Bonus.ipynb
BUYUK SAYILAR YASASI.ipynb
calling-llm-apis.ipynb
check.ipynb
cliff-walking.ipynb
Contacts
Cookies
creditcard.csv
credit_card.ipynb
custom-env.ipynb
custom_grid_dqn.zip
data
Data House Price -Bonus.ipynb
data-context-and-setup
data-law-of-large-number
data-query-the-db
data-travel-agency-database
data_back_to_school_query
Documents
dqn_cliffwalking_fast.zip
dqn_cliffwalking_model.zip
ecommerce.sqlite
e_devlet_eimza.log
IntelGraphicsProfiles
Jedi
KAGGLE YARISMA.ipynb
kaggle-house-prices
Links
Local Settings
MARKOV ZINCIRLERI.ipynb
Microsoft
NetHood
nlp_olist_ba

🧠 Artık ajanınızın ortamda hareket ettiğini görmelisiniz.

Eğer ajan **belirli eylemleri tekrarlayarak takılıp kalırsa** veya hedefe ulaşamazsa, muhtemelen **yeterince uzun eğitilmediği** içindir.

Eğitim adım sayısını artırmayı deneyin ve modeli yeniden eğitin — daha uzun eğitim genellikle ajanın daha iyi bir politika öğrenmesine yardımcı olur.